In [40]:
import pandas as pd
import glob
import os

RAW_DIR = '../../data/norming_study/raw/b_pilot/'

In [41]:
# load all half csvs (skip demographics)
half_files = glob.glob(os.path.join(RAW_DIR, '*_half.csv'))
raw = pd.concat([pd.read_csv(f) for f in half_files], ignore_index=True)

print(f'{len(half_files)} files loaded, {len(raw)} total rows')
raw.head()

10 files loaded, 160 total rows


,trial_type,subjectID,prolificID,studyID,sessionID,DEBUG,sliderOrder,half,targetValue,passed,...,vignette,abilityResponse,abilityRT,abilityDragged,willingnessResponse,willingnessRT,willingnessDragged,trialRT,trialIndex,suspicious
0,whyask-trial,y0fe40cx3r,NaN,NaN,NaN,0,WA,2,NaN,NaN,...,You are sitting at a restaurant with a friend ...,8.0,18615.0,True,16.0,15358.0,True,19914.0,16,False
1,whyask-trial,y0fe40cx3r,NaN,NaN,NaN,0,WA,2,NaN,NaN,...,An actor in your friend group has dropped out ...,30.0,8400.0,True,12.0,4059.0,True,14735.0,17,False
2,whyask-trial,y0fe40cx3r,NaN,NaN,NaN,0,WA,2,NaN,NaN,...,Your close friend has a sudden emergency appoi...,28.0,9183.0,True,69.0,5392.0,True,11608.0,18,False
3,whyask-trial,y0fe40cx3r,NaN,NaN,NaN,0,WA,2,NaN,NaN,...,You and some friends are at a hot wing restaur...,22.0,7092.0,True,9.0,5078.0,True,9268.0,19,False
4,whyask-trial,y0fe40cx3r,NaN,NaN,NaN,0,WA,2,NaN,NaN,...,Your friend's car has broken down on the side ...,25.0,5798.0,True,78.0,1618.0,True,7114.0,20,False


In [42]:
# split trials (attn checks removed from study)
trials = raw[raw['half'].isin([1, 2])].copy()
# attn = raw[raw['half'] == 'attn'].copy()

print(f'trials: {len(trials)} rows, {trials["subjectID"].nunique()} subjects')
print(f'subjects: {sorted(trials["subjectID"].unique())}')

trials: 75 rows, 5 subjects
subjects: ['9jt5xmqcz1', 'fkexq8dztx', 'ozppteowo7', 'wvwpa4yudu', 'y0fe40cx3r']


In [ ]:
# load demographics
demo_files = glob.glob(os.path.join(RAW_DIR, '*_demographics.csv'))
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

print(f'{len(demo)} demo rows')

5 demo rows


,subjectID,age,gender,race,education,totalDurationMs
0,ozppteowo7,100,Male,Asian,masters,606310
1,9jt5xmqcz1,100,Male,White,doctorate,722200
2,wvwpa4yudu,25,Male,Asian,bachelors,1692393
3,y0fe40cx3r,30,Male,Asian,masters,394573
4,fkexq8dztx,29,Male,Asian,doctorate,854208


In [44]:
# attn check summary — commented out (no attn checks in study)
# attn_cols = ['subjectID', 'itemID', 'abilityResponse', 'willingnessResponse', 'suspicious']
# if 'targetValue' in attn.columns:
#     attn_cols.insert(4, 'targetValue')
# print(attn[attn_cols])

In [45]:
# attn check filtering commented out (no attn checks in study)
# failed = attn[attn['suspicious'] == True]['subjectID'].unique()
# trials_clean = trials[~trials['subjectID'].isin(failed)].copy()
# demo_clean   = demo[~demo['subjectID'].isin(failed)].copy()

trials_clean = trials.copy()
demo_clean   = demo.copy()

print(f'kept {trials_clean["subjectID"].nunique()} subjects ({len(trials_clean)} trials)')

kept 5 subjects (75 trials)


In [46]:
# quick look at trial responses
print(trials[['subjectID', 'itemID', 'actionPhrase', 'abilityResponse', 'willingnessResponse', 'sliderOrder']].to_string())
print()
# avg time per trial
avg_rt = trials['trialRT'].mean() / 1000
print(f'avg trial RT: {avg_rt:.1f}s')

      subjectID itemID                                                                        actionPhrase  abilityResponse  willingnessResponse sliderOrder
34   ozppteowo7     74                                 give me your old laptop now that you have a new one            100.0                 50.0          AW
35   ozppteowo7    110                               call my landlord to complain about the heat being out            100.0                 10.0          AW
36   ozppteowo7    101                                     do a stand-up comedy set at my open mic tonight             99.0                 10.0          AW
37   ozppteowo7    106                                           help me paint my new bedroom this weekend             79.0                 41.0          AW
38   ozppteowo7     90                                       help me find a good therapist to start seeing             24.0                 88.0          AW
39   ozppteowo7    128                  tell the police yo

In [47]:
# save processed
PROC_DIR = '../../data/norming_study/processed/b_pilot/'
os.makedirs(PROC_DIR, exist_ok=True)

# drop trial-level cols from demo
trial_cols = ['trialIndex', 'itemID', 'actionPhrase', 'abilityResponse', 'abilityRT',
              'abilityDragged', 'willingnessResponse', 'willingnessRT', 'willingnessDragged',
              'trialRT', 'suspicious', 'targetValue', 'passed']
demo_out = demo_clean.drop(columns=[c for c in trial_cols if c in demo_clean.columns])

trials_clean.to_csv(os.path.join(PROC_DIR, 'trials.csv'), index=False)
demo_out.to_csv(os.path.join(PROC_DIR, 'demographics.csv'), index=False)

print(f'saved trials.csv ({len(trials_clean)} rows) and demographics.csv ({len(demo_out)} rows)')
print(f'demo cols: {list(demo_out.columns)}')

saved trials.csv (75 rows) and demographics.csv (5 rows)
demo cols: ['trial_type', 'subjectID', 'prolificID', 'studyID', 'sessionID', 'DEBUG', 'sliderOrder', 'half', 'age', 'gender', 'race', 'education', 'strategy', 'technicalIssues', 'feedback', 'visibilityChanges', 'totalDurationMs']
